# Charging Desert Analysis Notebook

In [ ]:
import sys 
from pathlib import Path 
import geopandas as gpd
import matplotlib.pyplot as plt 
import seaborn as sns

## Load metric and scored tract layers

In [ ]:
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT / "src"))
from config import GEOJSON_DIR, PROCESSED_DIR , CRS_WEB

deserts_gdf = gpd.read_file(GEOJSON_DIR / "charging_deserts.geojson")

metrics_gdf = gpd.read_parquet(PROCESSED_DIR / "tract_coverage_metrics.parquet")

print(f"Loaded {len(deserts_gdf)} tracts with CRS: {deserts_gdf.crs}")
deserts_gdf.head()

## Score distribution

In [ ]:
# Descriptive statistics for desert scores
deserts_gdf["desert_score"].describe()

In [ ]:
# Histogram of score values
x = deserts_gdf['desert_score']
plt.figure(figsize=(8,5))
plt.hist(x, color='skyblue', edgecolor='black', bins=20, alpha=0.7)
plt.axvline(50, color='orange', linestyle='--', label='Moderate (50)')
plt.axvline(75, color='red', linestyle='--', label='Severe (75)')
plt.xlabel('Desert Score (0-100)')
plt.ylabel('Number of Tracts')
plt.title("Desert Score Distribution Across California")
plt.grid(True, linestyle=':', alpha=0.5)
plt.show()


In [ ]:
# Count tracts by severity class
deserts_gdf['desert_score'].value_counts()

## Component Interpretation

In [ ]:
# Summary statistics of components
component_cols = [
    'comp_gas_access',
    'comp_pop_density',
    'comp_ev_gap',
    'comp_dcfc_distance'
]

deserts_gdf[comonent_cols].describe().round(3)

In [ ]:
# Box plots of distributions
plt.figure(figsize=(10,5))
sns.boxplot(data=deserts_gdf[component_cols], palette='Set2')
plt.xticks(
    ticks=range(4),
    labels=['Gas Access', 'Pop Density', 'EV Gap', 'DCFC Distance']
)
plt.ylabel("Normalized Score (0-1)")
plt.title("Distribution of Desert Score Components")
plt.grid(True, linestyle=':', alpha=0.5)
plt.show()

In [ ]:
# Create a 2x2 subplot layout for the 4 component scatter plots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

components = [
    ('comp_gas_access', 'Gasoline Access Component', axes[0, 0]),
    ('comp_pop_density', 'Population Density Component', axes[0, 1]),
    ('comp_ev_gap', 'EV Gap Component (1 - Supply)', axes[1, 0]),
    ('comp_dcfc_distance', 'DCFC Distance Component', axes[1, 1])
]

for col, title, ax in components:
    sns.scatterplot(
        data=deserts_gdf, 
        x=col, 
        y='desert_score', 
        hue='severity_score', 
        palette={'adequate': 'gray', 'moderate_desert': 'orange', 'severe_desert': 'red'},
        alpha=0.4, 
        s=15,
        ax=ax
    )
    ax.set_title(f'Desert Score vs. {title}')
    ax.set_xlabel('Component Value (0 - 1)')
    ax.set_ylabel('Final Desert Score (0 - 100)')
    ax.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()


## Map outputs

In [ ]:
# Choropleth of desert score
import matplotlib.pyplot as plt 

fig, ax = plt.subplots(1,1, figsize=(10,10))

desert_gdf.plot(
    column='desert_score',
    cmap="YlORd",
    legend=True,
    legend_kwds={'label': 'Charging Desert Score (0-100)', 'orientation': 'vertical'},
    ax=ax,
    edgecolor='none'
)
ax.set_title("California EV Charging Desert Scores by Census Tract", fontsize=14)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# Custom color mapping for severity classes
color_dict = {
    "adequate": "#e0e0e0",         # Light gray
    "moderate_desert": "#f39c12",   # Orange
    "severe_desert": "#c0392b"      # Dark Red
}

deserts_gdf.plot(
    column="severity_score",
    color=deserts_gdf["severity_score"].map(color_dict),
    legend=True,
    ax=ax,
    edgecolor="none"
)

ax.set_title("California EV Charging Desert Severity Classification", fontsize=14)
ax.set_axis_off()
plt.tight_layout()
plt.show()


In [ ]:
# Inspect top 15 severe tracts
top_15 = deserts_gdf['desert_score'].value_counts(ascending=False).head(15)
top_15


## Demographic Context

In [ ]:
# Compare desert score against median household
plt.figure(figsize=(10,5))

# Filter out tracts where income is missing/null
valid_income = deserts_gdf[desert_gdf['median_income'].notna()]

# Scatter plot
sns.scatterplot(
    data=valid_income_gdf,
    x='median_income',
    y='desert_score',
    hue='severity_score',
    palette={'adequate': 'gray', 'moderate_desert': 'orange', 'severe_desert': 'red'},
    alpha=0.4,
    s=15
)

# Trend line
sns.regplot(
    data=valid_income,
    x='median_income',
    y='desert_score',
    scatter=False,
    color='black',
    line_kws={'linestyle': '--', 'linewidth': 1.5}
)
plt.xlabel("Median Household Income")
plt.ylabel("Charging Desert Score (0-100)")
plt.title("Median Household Income vs Charging Desert Score")
plt.show()


In [ ]:
# Compare desert score against population density
plt.figure(figsize=(8,5))

# Filter out where population is missing
valid_pop = deserts_gdf[deserts_gdf['population_density'].notna()]

# Scatter plot 
sns.scatterplot(
    data=valid_income_gdf,
    x='population_density',
    y='desert_score',
    hue='severity_score',
    palette={'adequate': 'gray', 'moderate_desert': 'orange', 'severe_desert': 'red'},
    alpha=0.4,
    s=15
)

# Trend line
sns.regplot(
    data=valid_income,
    x='population_density',
    y='desert_score',
    scatter=False,
    color='black',
    line_kws={'linestyle': '--', 'linewidth': 1.5}
)
plt.xlabel("Population Density")
plt.ylabel("Charging Desert Score (0-100)")
plt.title("Population Density vs Charging Desert Score")
plt.show()

## Export

In [ ]:
geojson_path = GEOJSON_DIR / "charging_deserts.geojson"

deserts_web_gdf = deserts_gdf.to_crs(CRS_WEB)

deserts_web_gdf.to_file(geojson_path, driver='GEOJSON')
print(f"Exported Charging Deserts GeoJSON to {geojson_path}")